> **Note:** This notebook requires run outputs that are not included in the published repository. To reproduce, re-run the pipeline with the appropriate configuration to generate the required analysis DataFrames.
>
> This notebook references DOWNSAMPLE_JUN03_* downsampled runs (post-adjacency-fix sweep) that must be regenerated via the pipeline.


In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import scipy.stats as stats 

# enable latex plotting 
plt.rc('text', usetex=True)
plt.rc('font', family='serif')


In [2]:
from constants import CURRENT_DF

In [3]:
import re
from glob import glob

RUNS_DIR = "../../runs/icar_icar/simulated_False/ahl_True/covariates_True"

# Post-adjacency-fix downsampling batch. Each run dir is named
#   DOWNSAMPLE_JUN03_<FAMILY>_<FRAC>_<TIMESTAMP>/
# and a run only writes its analysis_df_*.csv AFTER passing the r_hat < 1.1
# convergence check (icar_model.validate_results). Globbing on that file
# therefore discovers exactly the converged runs and silently skips any that
# aborted on non-convergence -- which is then reported explicitly below so the
# downstream correlations are never computed on a partial sweep.
ANNOT_PREFIX = "DOWNSAMPLE_JUN03_ANNOT"
ALL_PREFIX = "DOWNSAMPLE_JUN03_ALL"


def _discover(prefix):
    """frac -> (timestamp, analysis_df_path) for every CONVERGED run under
    `prefix`. Most recent timestamp wins if a frac was run more than once."""
    runs = {}
    for d in glob(f"{RUNS_DIR}/{prefix}_*"):
        m = re.search(rf"{re.escape(prefix)}_([0-9.]+)_(\d{{8}}-\d{{4}})$", d)
        if not m:
            continue
        frac, ts = float(m.group(1)), m.group(2)
        adf = [p for p in glob(f"{d}/analysis_df_*.csv")
               if "analysis_df_describe" not in p]
        if not adf:
            continue  # no analysis_df -> did not pass the convergence gate
        if frac not in runs or ts > runs[frac][0]:
            runs[frac] = (ts, adf[0])
    return runs


def _load(prefix, family_label):
    runs = _discover(prefix)
    out = {}
    for frac in sorted(runs, reverse=True):  # 0.5 (2x) ... 0.02 (50x)
        n = round(1 / frac)
        key = f"downsampled_{family_label}{n}x"
        out[key] = pd.read_csv(runs[frac][1])
        print(f"  {key:24s} frac={frac:<7} {runs[frac][1].split('/')[-2]}")
    return out


print("ANNOTATED-ONLY downsampling runs:")
annotated_only_runs_to_compare = _load(ANNOT_PREFIX, "")
print("\nALL-IMAGES downsampling runs:")
all_downsampled_runs_to_compare = _load(ALL_PREFIX, "all_")

# Flag ratios whose run did not converge (no analysis_df written).
EXPECTED_ANNOT = {2, 5, 10, 20, 50}
EXPECTED_ALL = {2, 3, 4, 5, 10}
missing_annot = sorted(EXPECTED_ANNOT - {round(1 / f) for f in _discover(ANNOT_PREFIX)})
missing_all = sorted(EXPECTED_ALL - {round(1 / f) for f in _discover(ALL_PREFIX)})
if missing_annot:
    print(f"\n*** MISSING (non-converged) annotated-only ratios: "
          f"{[f'{n}x' for n in missing_annot]} ***")
if missing_all:
    print(f"\n*** MISSING (non-converged) all-images ratios: "
          f"{[f'{n}x' for n in missing_all]} ***")

ANNOTATED-ONLY downsampling runs:
  downsampled_2x           frac=0.5     DOWNSAMPLE_JUN03_ANNOT_0.5_20260603-1352
  downsampled_5x           frac=0.2     DOWNSAMPLE_JUN03_ANNOT_0.2_20260603-1438


  downsampled_10x          frac=0.1     DOWNSAMPLE_JUN03_ANNOT_0.1_20260603-1523
  downsampled_20x          frac=0.05    DOWNSAMPLE_JUN03_ANNOT_0.05_20260603-1607
  downsampled_50x          frac=0.02    DOWNSAMPLE_JUN03_ANNOT_0.02_20260603-1658

ALL-IMAGES downsampling runs:
  downsampled_all_2x       frac=0.5     DOWNSAMPLE_JUN03_ALL_0.5_20260604-1627
  downsampled_all_4x       frac=0.25    DOWNSAMPLE_JUN03_ALL_0.25_20260603-1948
  downsampled_all_5x       frac=0.2     DOWNSAMPLE_JUN03_ALL_0.2_20260603-2035

*** MISSING (non-converged) all-images ratios: ['3x', '10x'] ***


### All-images downsampling: convergence boundary (post-adjacency-fix)

The all-images downsampling family is **truncated at the highest ratio whose fit
converges** ($\hat R < 1.1$ on every parameter), exactly as in the original
paper, which dropped the 20$\times$ and 50$\times$ all-images runs for the same
reason.

Post-fix (`DOWNSAMPLE_JUN03_ALL_*`), the converged ratios are **2$\times$, 4$\times$,
5$\times$** (correlations 0.95 / 0.83 / 0.84 vs. the full-data reference). The
3$\times$ and 10$\times$ runs repeatedly fail convergence on `p_y[1085]`
(a single structurally hard-to-identify tract) across multiple seeds
(201/202/205 then 211/212/215), with $\hat R \approx 1.25$–1.44 — i.e. it is a
persistent identifiability wall under heavy all-image thinning, **not** seed
bad-luck. 10$\times$ was reported as 0.53 in the published paper; post-fix it no
longer reliably converges, so the all-images sweep now stops at 5$\times$.

Note the post-fix numbers tell a *stronger* stability story than the published
ones (5$\times$ moved 0.69 $\to$ 0.84): the adjacency fix made predictions hold
up better under sparse image data. The annotated-only family (which carries far
more identifying information per area) converges at every ratio out to 50$\times$
and stays at 0.97–0.99 throughout.

The cell above discovers runs by prefix glob and only loads ratios that wrote an
`analysis_df` (i.e. passed the convergence gate), printing any missing ratio so a
partial sweep can never be silently reported.

In [4]:
# (ANNOTATED-ONLY and ALL-IMAGES runs are both discovered in the cell above
# via prefix glob, replacing the previous hard-coded per-run paths.)

In [5]:
# annotated-only runs discovered above:
list(annotated_only_runs_to_compare.keys())

['downsampled_2x',
 'downsampled_5x',
 'downsampled_10x',
 'downsampled_20x',
 'downsampled_50x']

In [6]:
# all-images runs discovered above:
list(all_downsampled_runs_to_compare.keys())

['downsampled_all_2x', 'downsampled_all_4x', 'downsampled_all_5x']

In [7]:
reference_run = pd.read_csv(CURRENT_DF)

In [8]:
reference_run.head()

,BoroName,BoroCT2020,NTAName,CDTANAME,PUMA,empirical_estimate,p_y,p_y_CI_lower,p_y_CI_upper,n_images_by_area,...,dep_moderate_1_area,dep_moderate_1_frac,dep_moderate_2_area,dep_moderate_2_frac,GEOID,sewer_backup_311c,street_flooding_311c,catch_basin_clogged/flooding_311c,manhole_overflow_311c,highway_flooding_311c
0,Manhattan,1000100,The Battery-Governors Island-Ellis Island-Libe...,MN01 Financial District-Tribeca (CD 1 Equivalent),4121,NaN,0.059687,1.652865e-312,1.000000,0,...,0.000000,0.000000,0.000000,0.000000,36061000100,0,0,0,0,0
1,Manhattan,1000201,Chinatown-Two Bridges,MN03 Lower East Side-Chinatown (CD 3 Equivalent),4103,0.000000,0.002002,2.326344e-05,0.012153,345,...,0.000000,0.000000,0.000000,0.000000,36061000201,0,0,0,0,0
2,Manhattan,1000600,Chinatown-Two Bridges,MN03 Lower East Side-Chinatown (CD 3 Equivalent),4103,0.002217,0.008113,5.828994e-04,0.026918,902,...,22123.775465,0.008566,28743.307693,0.011129,36061000600,0,1,0,0,0
3,Manhattan,1001401,Lower East Side,MN03 Lower East Side-Chinatown (CD 3 Equivalent),4103,0.000000,0.000983,1.465489e-05,0.005964,259,...,0.000000,0.000000,0.000000,0.000000,36061001401,1,0,0,0,0
4,Manhattan,1001402,Lower East Side,MN03 Lower East Side-Chinatown (CD 3 Equivalent),4103,0.000000,0.000122,1.964080e-06,0.000786,705,...,3811.632650,0.003108,7439.195282,0.006067,36061001402,0,0,0,0,0


In [9]:
# compute correlation between reference run and downsampled runs

from typing import Any
downsampled_pearson_corrs = []

# align reference and downsampled runs on the area-id key before correlating,
# so tracts are matched by id rather than by (fragile) row position
_id_key = 'BoroCT2020' if 'BoroCT2020' in reference_run.columns else 'GEOID'

for i, downsampled_run in enumerate(annotated_only_runs_to_compare.values()):
    # compute correlation between reference run and downsampled run on p_y column
    _merged = reference_run[[_id_key, 'p_y']].merge(
        downsampled_run[[_id_key, 'p_y']], on=_id_key, how='inner', suffixes=('_ref', '_ds')
    )
    corr = np.corrcoef(_merged['p_y_ref'], _merged['p_y_ds'])[0, 1]
    pearson_corr = stats.pearsonr(_merged['p_y_ref'], _merged['p_y_ds'])
    downsampled_pearson_corrs.append(pearson_corr.statistic)

    print(f"Correlation between reference run and {list(annotated_only_runs_to_compare.keys())[i]} run: {corr}")





Correlation between reference run and downsampled_2x run: 0.9886803635577605
Correlation between reference run and downsampled_5x run: 0.9751081047203258
Correlation between reference run and downsampled_10x run: 0.9700778982427262
Correlation between reference run and downsampled_20x run: 0.9713866743908964
Correlation between reference run and downsampled_50x run: 0.9681509062555933


In [10]:
downsampled_all_pearson_corrs = []

# align on the area-id key (id-based join) instead of relying on row order
_id_key = 'BoroCT2020' if 'BoroCT2020' in reference_run.columns else 'GEOID'

for i, downsampled_run in enumerate(all_downsampled_runs_to_compare.values()):
    _merged = reference_run[[_id_key, 'p_y']].merge(
        downsampled_run[[_id_key, 'p_y']], on=_id_key, how='inner', suffixes=('_ref', '_ds')
    )
    pearson_corr = stats.pearsonr(_merged['p_y_ref'], _merged['p_y_ds'])
    downsampled_all_pearson_corrs.append(pearson_corr.statistic)

    print(f"Correlation between reference run and {list(all_downsampled_runs_to_compare.keys())[i]} run: {pearson_corr.statistic}")

Correlation between reference run and downsampled_all_2x run: 0.9537098158082162
Correlation between reference run and downsampled_all_4x run: 0.8292521754050955
Correlation between reference run and downsampled_all_5x run: 0.8392091570343754
